Este modulo realiza un append de la data de la ejecución de hoy (04/08/2026) en un volumen de databricks.
La data se lee desde el mismo volumen pero podría provenir de una fuente externa (Cómo S3 o Google Cloud Storage) y aplicaría una solución equivalente.

El rol de esta etapa es unificar la información extraida de distintas fuentes y mantener un registro historico de la información de todas las cargas la cuál este particionada por mes y año.

In [0]:
dataset_path = "/Volumes/iol_challenge/bronze/transaction_data/data_to_land/2026/08/04/challenge-iol-data-set.csv"
target_path = "iol_challenge.bronze.raw_ingestion"

df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(dataset_path)
    )

Enriquecemos la data de bronce con checkeos de calidad, columnas para particionamiento y el timestamp de ingesta

In [0]:
from pyspark.sql.functions import year, month, col, now, array_remove, array, when, expr, length, size, to_json, struct, lit

(
    df
        .withColumn("data_type", lit("transaction"))
        .withColumn("data", to_json(struct(*df.columns), options={"ignoreNullFields": "true"}))
        .withColumn("anio", year(col("fecha")))
        .withColumn("mes", month(col("fecha")))
        .withColumn("dia", month(col("fecha")))
        .withColumn("timestamp_ejecucion", now())
        .withColumn("errores_calidad", array_remove(
                array(
                    when(col("id_transaccion").isNull(), "ID_DE_TRANSACCION_NULO"),
                    when(length(col("id_transaccion")) != 14, "LONGITUD_DE_TRANSACCION_INVALIDA"),
                    when(col("fecha") > expr("current_timestamp()"), "FECHA_INVALIDA")
                ),
                None 
            )
        ).withColumn("tiene_errores_calidad",
            col("errores_calidad").isNotNull()
        ).select(
            col("data_type"),
            col("data"),
            col("dia"),
            col("anio"),
            col("mes"),
            col("timestamp_ejecucion"),
            col("errores_calidad"),
            col("tiene_errores_calidad"),
        )
        .write
        .format("delta")      
        .mode("append")
        .partitionBy("anio", "mes", "dia")
        .saveAsTable(target_path)
)

Verificación de la generación de la tabla con información de la capa bronce ya generado

In [0]:
%sql
SELECT * FROM iol_challenge.bronze.raw_ingestion LIMIT 10;

data_type,data,dia,anio,mes,timestamp_ejecucion,errores_calidad,tiene_errores_calidad
transaction,"{""fecha"":""2026-02-19T17:35:16.000Z"",""tipoTran"":""Compra"",""id_cliente"":""CLI1CC7542B"",""descripcion_titulo"":""Cedear Microsoft Corp."",""moneda"":""ARS"",""simbolo_titulo"":""MSFT"",""cantidad"":24,""precio"":19481.5155,""id_transaccion"":""TXNhxthv3a3zmf8m"",""origen"":""Sitio Web Desktop""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-23T23:43:53.000Z"",""tipoTran"":""Compra"",""id_cliente"":""CLI6691FC58"",""descripcion_titulo"":""Bono Rep. Argentina Usd Step Up 2030"",""moneda"":""ARS"",""simbolo_titulo"":""AL30"",""cantidad"":419,""precio"":856.398,""id_transaccion"":""TXNj7xvg0fn9xuy4"",""origen"":""Sitio Web Desktop""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-25T02:33:14.000Z"",""tipoTran"":""Compra"",""id_cliente"":""CLI0D41FC57"",""descripcion_titulo"":""Cedear Proshares Short S&P500"",""moneda"":""ARS"",""simbolo_titulo"":""SH"",""cantidad"":21,""precio"":6553.7541,""id_transaccion"":""TXN1ibljh75lxo6q"",""origen"":""App Mobile""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-27T05:04:53.000Z"",""tipoTran"":""Venta"",""id_cliente"":""CLI032865E4"",""descripcion_titulo"":""BONO REP ARG AJ CER V30/06/28"",""moneda"":""ARS"",""simbolo_titulo"":""TZX28"",""cantidad"":24,""precio"":2.9034,""id_transaccion"":""TXNxvf1t2tala753"",""origen"":""App Mobile""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-06T14:10:25.000Z"",""tipoTran"":""Compra"",""id_cliente"":""CLI359D552D"",""descripcion_titulo"":""Cedear Amazon.Com, Inc"",""moneda"":""ARS"",""simbolo_titulo"":""AMZN"",""cantidad"":1,""precio"":2336.7974,""id_transaccion"":""TXNlc58drc11ertj"",""origen"":""App Mobile""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-07T00:43:29.000Z"",""tipoTran"":""Compra"",""id_cliente"":""CLIBF736BBA"",""descripcion_titulo"":""Cedear S&P Global Inc."",""moneda"":""ARS"",""simbolo_titulo"":""SPGI"",""cantidad"":1,""precio"":14579.601,""id_transaccion"":""TXNmvihcwi64ciyh"",""origen"":""Sitio Web Desktop""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-26T20:44:29.000Z"",""tipoTran"":""Compra"",""id_cliente"":""CLI772B0115"",""descripcion_titulo"":""Bono Rep. Argentina Usd Step Up 2030"",""moneda"":""ARS"",""simbolo_titulo"":""AL30"",""cantidad"":24,""precio"":873.2334,""id_transaccion"":""TXNe7ur23gdppq0y"",""origen"":""App Mobile""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-14T15:02:18.000Z"",""tipoTran"":""Venta"",""id_cliente"":""CLI32378867"",""descripcion_titulo"":""Grupo Financiero Galicia S.A"",""moneda"":""ARS"",""simbolo_titulo"":""GGAL"",""cantidad"":62,""precio"":6920.5535,""id_transaccion"":""TXN4dpja1wj0tpac"",""origen"":""App Mobile""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-28T09:49:17.000Z"",""tipoTran"":""Venta"",""id_cliente"":""CLI9C01D3D3"",""descripcion_titulo"":""Bono del Tesoro Boncer Vto 31/03/2026"",""moneda"":""ARS"",""simbolo_titulo"":""TZXM6"",""cantidad"":199566,""precio"":2.0961,""id_transaccion"":""TXN0l7253j2d54i3"",""origen"":""Sitio Web Desktop""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false
transaction,"{""fecha"":""2026-02-09T23:50:27.000Z"",""tipoTran"":""Venta"",""id_cliente"":""CLI60ECF1AF"",""descripcion_titulo"":""Cedear Asml Holding Nv"",""moneda"":""ARS"",""simbolo_titulo"":""ASML"",""cantidad"":3,""precio"":14489.7937,""id_transaccion"":""TXNll4zklookep7y"",""origen"":""App Mobile""}",2,2026,2,2026-08-05T05:58:01.342Z,null,false


In [0]:
%sql
WITH freq AS (
    SELECT get_json_object(data, '$.id_transaccion'), COUNT(*) as cantidad FROM iol_challenge.bronze.raw_ingestion WHERE data_type = 'transaction'  GROUP BY get_json_object(data, '$.id_transaccion')
) SELECT COUNT(*), cantidad FROM freq GROUP BY cantidad;

COUNT(*),cantidad
100000,1


In [0]:
%sql
SELECT COUNT(*), tiene_errores_calidad FROM iol_challenge.bronze.raw_ingestion GROUP BY tiene_errores_calidad;

COUNT(*),tiene_errores_calidad
100000,false
